<a href="https://colab.research.google.com/github/trainocate-japan/openai_api_app/blob/main/chapter2/%E7%AC%AC2%E7%AB%A0%20Chat%20Completions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAI API利用のための準備

In [ ]:
# openaiパッケージのインストール(colabはインストール済み)
# !pip install openai

In [ ]:
# インストール済みパッケージの確認
!pip freeze | grep openai

In [ ]:
# oepnaiパッケージのインポート
from openai import OpenAI

In [ ]:
# APIキーを設定
client = OpenAI(api_key="your api key")

# chat completions APIの利用

In [ ]:
# メソッドを呼び出し、応答を得る
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "こんにちは"}
        ]
    )

In [ ]:
# chat completionsの表示
print(response.choices[0].message.content)

# 対話履歴の書き出し(保存)

In [ ]:
# jsonパッケージをインポート
import json

messages = [{"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "こんにちは。私は猫が好きです。"}
           ]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
    )

# 入力したメッセージと応答を一つのリストにする
messages.append({"role": "assistant", "content": response.choices[0].message.content})

# 対話履歴をhistory.jsonに保存
with open("history.json", mode="w") as f:
    json.dump(messages, f)

# 対話履歴の読み込み(入力)


In [ ]:
# history.jsonを読み込む
with open("history.json", mode="r") as f:
    messages = json.load(f)

In [ ]:
# メッセージを追加し、回答を表示する
messages.append({"role": "user", "content": "世界に何種類いますか"})

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
    )

print(response.choices[0].message.content)

# 入力された文章の要約

In [ ]:
# 入力する文章の読み込み
with open("/content/走れメロス(青空文庫).txt") as f:
  data = f.read()
  print(data)
  print(type(data))

In [ ]:
# 読み込んだ文章を要約する
response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "次の文章を100文字以内で要約してください。\n"
    + "###\n"
    + data}
  ]
)

print(response.choices[0].message.content)

# トークンをマネジメントする

In [ ]:
# tiktokenパッケージのインストール（colabはインストール済み）
# !pip install tiktoken

In [ ]:
# インストール済みパッケージの確認
!pip freeze | grep tiktoken

In [ ]:
# tiktokenパッケージのインポート
import tiktoken

In [ ]:
# モデルのトークン化手法を確認
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

In [ ]:
#トークン数の計算
num_tokens = len(encoding.encode("おはようございます"))
print(num_tokens)

# Webアプリケーション実装

In [ ]:
# streamlit関連パッケージのインストール
!pip install streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!which cloudflared && cloudflared --version

In [ ]:
# app.py を作成
%%writefile app.py

In [ ]:
from google.colab import files
files.view('app.py')

In [ ]:
# app.pyの中身を実装
# ユーザーからの入力をそのまま返す
%%writefile app.py
import streamlit as st

st.title("Hello Streamlit")

user_input = st.text_input("あなたのメッセージを入力してください:")

if st.button("送信"):
    st.text_area("ボットの応答", user_input)

In [ ]:
# streamlitのrunコマンドでapp.pyを立ち上げ、localtunnelを用いてアプリ公開
# !streamlit run app.py & sleep 3 && npx localtunnel --port 8501

In [ ]:
# streamlitのrunコマンドでapp.pyを立ち上げ、localtunnelを用いてアプリ公開
!streamlit run app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.fileWatcherType none \
  > /tmp/st.log 2>&1 &
!for i in {1..60}; do curl -fsS http://localhost:8501/healthz && echo "Streamlit is up" && break || sleep 1; done

# トンネル起動（ログにURLが出る）
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate > /tmp/cf.log 2>&1 &

# URLがログに出るまで最大60秒待って抽出
!for i in {1..60}; do \
  URL=$(grep -o "https://[0-9a-z.-]*trycloudflare.com" -m 1 /tmp/cf.log); \
  if [ -n "$URL" ]; then echo "PUBLIC URL: $URL"; break; fi; \
  sleep 1; \
done

In [ ]:
# app.pyの中身を実装
# ユーザーからの入力に対する反応を表示する
%%writefile app.py
from openai import OpenAI
import streamlit as st

client = OpenAI(api_key="your api key")

st.title("シンプルなチャットボット")
user_input = st.text_input("あなたのメッセージを入力してください:")

if st.button("送信"):
    response = client.chat.completions.create(
        model= "gpt-4o-mini",
        messages=[{"role": "system", "content": "You are a helpful assistant."},
                  {"role": "user", "content": user_input}
                  ]
        )

    st.text_area("ボットの応答", response.choices[0].message.content)

In [ ]:
# streamlitのrunコマンドでapp.pyを立ち上げ、localtunnelを用いてアプリ公開
!streamlit run app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.fileWatcherType none \
  > /tmp/st.log 2>&1 &
!for i in {1..60}; do curl -fsS http://localhost:8501/healthz && echo "Streamlit is up" && break || sleep 1; done

# トンネル起動（ログにURLが出る）
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate > /tmp/cf.log 2>&1 &

# URLがログに出るまで最大60秒待って抽出
!for i in {1..60}; do \
  URL=$(grep -o "https://[0-9a-z.-]*trycloudflare.com" -m 1 /tmp/cf.log); \
  if [ -n "$URL" ]; then echo "PUBLIC URL: $URL"; break; fi; \
  sleep 1; \
done